# Bitcoin: classification

Section 3.2 and Table 3. For each scenario we hold out a stratified 20% test partition. On the remaining 80% a Random Forest, an MLP and an RBF SVM are tuned by stratified five-fold cross-validation on macro-F1. SMOTE sits inside the pipeline, so it only ever sees training folds. The tuned pipelines are then scored once on the test partition.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import json

import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from bfx.bitcoin import FEATURES
from bfx.modeling import SEED, scores

DATA = ROOT / "data" / "bitcoin" / "scenarios"
RESULTS = ROOT / "results" / "bitcoin"
MODELS = ROOT / "models" / "bitcoin"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

In [ ]:
def candidates():
    smote = SMOTE(random_state=SEED)
    return {
        "RF": (Pipeline([("smote", smote), ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1))]),
               {"clf__n_estimators": [200, 500], "clf__max_depth": [20, None], "clf__min_samples_leaf": [1, 2]}),
        "MLP": (Pipeline([("scale", StandardScaler()), ("smote", smote),
                          ("clf", MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED))]),
                {"clf__hidden_layer_sizes": [(64,), (128, 64)], "clf__alpha": [1e-4, 1e-3]}),
        "SVM": (Pipeline([("scale", StandardScaler()), ("smote", smote),
                          ("clf", SVC(kernel="rbf", probability=True, random_state=SEED))]),
                {"clf__C": [1, 10], "clf__gamma": ["scale"]}),
    }


def split(k):
    data = pd.read_csv(DATA / f"scenario_{k}.csv")
    return train_test_split(data[FEATURES], data["y"], test_size=0.2, stratify=data["y"], random_state=SEED)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows, tuning = [], {}
for k in (1, 2, 3):
    X_train, X_test, y_train, y_test = split(k)
    for name, (pipe, grid) in candidates().items():
        search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=-1).fit(X_train, y_train)
        model = search.best_estimator_
        s = scores(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1])
        rows.append({"scenario": k, "model": name, **s})
        best = search.cv_results_["rank_test_score"].argmin()
        tuning[f"{k}/{name}"] = {"params": search.best_params_,
                                 "cv_f1_macro_mean": search.cv_results_["mean_test_score"][best],
                                 "cv_f1_macro_std": search.cv_results_["std_test_score"][best]}
        joblib.dump(model, MODELS / f"scenario_{k}_{name}.joblib")

results = pd.DataFrame(rows)
results.to_csv(RESULTS / "table3_classification.csv", index=False)
_ = (RESULTS / "table3_tuning.json").write_text(json.dumps(tuning, indent=2, default=float))

## Table 3

Macro-averaged precision, recall and F1, and accuracy, on the held-out test partition.

In [ ]:
table3 = results.set_index(["scenario", "model"])[["precision", "recall", "f1", "accuracy"]]
table3.round(2)

Cross-validated macro-F1 of the selected Random Forest on the training partition of Scenario 2 (mean and standard deviation over the five folds):

In [ ]:
t = tuning["2/RF"]
print(f"{t['cv_f1_macro_mean']:.3f} ± {t['cv_f1_macro_std']:.3f}   params: {t['params']}")

Operating figures used later in Table 10 (Scenario 2, Random Forest):

In [ ]:
results.query("scenario == 2 and model == 'RF'")[["balanced_accuracy", "roc_auc", "fpr", "tp", "fp", "tn", "fn"]]